## Detector IoU on the held-out CRIC test crops

IoU (intersection over union) is the overlap area of a predicted box and a true box divided by the area they cover together: 1 means identical, 0 means no overlap. mAP50 and mAP50-95 already use it. This notebook reports the plain average.

Run on the 1,335 held-out test crops that have a usable box, for the two stage-2 detectors (plain and oversampled), at 224 px and confidence 0.25. The "true" boxes are the ones we derived by segmentation (see `bounding_boxes_and_detector_training.ipynb`), not pathologist boxes, so this measures agreement with our own segmentation.

Inputs: the `cervigrade-cell-boxes` dataset (`cric_boxes.csv` and the detector `.pt` files) and the CRIC crops dataset. Kaggle notebook name: "IoU notebook".


In [1]:
!pip install -q ultralytics

In [2]:
# ---- IoU of the detector's boxes against the derived boxes, on the held-out TEST crops ----
# Needs: the cervigrade-cell-boxes dataset (boxes + weights) and the cervigrade-cric-dataset (the crops) attached as Inputs,
# and ultralytics installed (!pip install -q ultralytics).
import os, glob, random
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO

INPUT_ROOT = os.environ.get("IOU_INPUT_ROOT", "/kaggle/input")
CONF = 0.25                      # the same cutoff the app uses

# 1. The box file and the weights
box_csv = next(p for p in glob.glob(f"{INPUT_ROOT}/**/cric_boxes.csv", recursive=True))
weights = {
    "stage 2, plain":       next(iter(glob.glob(f"{INPUT_ROOT}/**/stage2_severity_best.pt", recursive=True)), None),
    "stage 2, oversampled": next(iter(glob.glob(f"{INPUT_ROOT}/**/stage2_severity_oversampled*.pt", recursive=True)), None),
}
weights = {k: v for k, v in weights.items() if v}
print("boxes:", box_csv); print("weights:", weights)

# 2. The SAME test split as the training notebook: whole slides, seed 42, 70/15/15 (computed on all rows, as there)
def split_by_group_global(df, seed=42, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train, n_val = round(n * train_frac), round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]: group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]: group_split[g] = "val"
    for g in groups[n_train + n_val:]: group_split[g] = "test"
    return df["group"].map(group_split)

cb = pd.read_csv(box_csv)
cb["group"] = cb["filepath"].apply(lambda p: os.path.basename(p).split("_")[0])
cb["split"] = split_by_group_global(cb)
test = cb[(cb["split"] == "test") & (cb["ok"] == True)].reset_index(drop=True)
print(f"Test crops with a usable box: {len(test)}  (the training notebook wrote 1,335)")

# the csv stores the path the crops had when the boxes were made; find them again if the mount point moved
def resolve(p):
    if os.path.exists(p):
        return p
    hits = glob.glob(f"{INPUT_ROOT}/**/{os.path.basename(p)}", recursive=True)
    return hits[0] if hits else None
if not os.path.exists(test["filepath"].iloc[0]):
    index = {}
    for f in glob.glob(f"{INPUT_ROOT}/**/*.png", recursive=True) + glob.glob(f"{INPUT_ROOT}/**/*.jpg", recursive=True):
        index[os.path.basename(f)] = f
    test["filepath"] = test["filepath"].apply(lambda p: index.get(os.path.basename(p)))
test = test.dropna(subset=["filepath"]).reset_index(drop=True)
print("Crops found on disk:", len(test))

# 3. IoU
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]); ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def run(model, df, batch=64):
    rows = []
    for s in range(0, len(df), batch):
        chunk = df.iloc[s:s + batch]
        imgs = [cv2.imread(p) for p in chunk["filepath"]]
        res = model.predict(imgs, imgsz=224, conf=CONF, verbose=False)
        for (_, r), pr in zip(chunk.iterrows(), res):
            gt = (r["x_min"], r["y_min"], r["x_max"], r["y_max"])
            if pr.boxes is None or len(pr.boxes) == 0:
                rows.append((r["filepath"], r["label"], 0, 0.0, 0.0)); continue
            xyxy = pr.boxes.xyxy.cpu().numpy(); conf = pr.boxes.conf.cpu().numpy()
            ious = np.array([iou(gt, b) for b in xyxy])
            rows.append((r["filepath"], r["label"], len(xyxy), float(ious.max()), float(ious[conf.argmax()])))
    return pd.DataFrame(rows, columns=["filepath", "label", "n_boxes", "best_iou", "top_box_iou"])

summary = []
for name, wpath in weights.items():
    model = YOLO(wpath)
    out = run(model, test)
    out.to_csv(f"/kaggle/working/iou_{name.replace(' ', '_').replace(',', '')}.csv" if os.path.isdir("/kaggle/working") else f"iou_{name.replace(' ', '_').replace(',', '')}.csv", index=False)
    found = out[out["best_iou"] > 0]
    summary.append({
        "detector": name, "test crops": len(out),
        "crops with no box": int((out["n_boxes"] == 0).sum()),
        "boxes per crop": round(out["n_boxes"].mean(), 2),
        "mean IoU (best box, all crops)": round(out["best_iou"].mean(), 3),
        "mean IoU (best box, boxed crops)": round(found["best_iou"].mean(), 3),
        "mean IoU (highest-confidence box)": round(out["top_box_iou"].mean(), 3),
        "share IoU >= 0.50": round((out["best_iou"] >= 0.50).mean(), 3),
        "share IoU >= 0.75": round((out["best_iou"] >= 0.75).mean(), 3),
        "median IoU": round(out["best_iou"].median(), 3),
    })
    by_class = out.groupby("label")["best_iou"].agg(["count", "mean", lambda s: (s >= 0.5).mean()]).round(3)
    by_class.columns = ["crops", "mean IoU", "share >= 0.50"]
    print(f"\n{name}: IoU by class"); print(by_class.to_string())

print("\n", pd.DataFrame(summary).T.to_string())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
boxes: /kaggle/input/datasets/wagnermushayija/cervigrade-cell-boxes/cric_boxes.csv
weights: {'stage 2, plain': '/kaggle/input/datasets/wagnermushayija/cervigrade-cell-boxes/stage2_severity_best.pt', 'stage 2, oversampled': '/kaggle/input/datasets/wagnermushayija/cervigrade-cell-boxes/stage2_severity_oversampled_best.pt'}
Test crops with a usable box: 1335  (the training notebook wrote 1,335)
Crops found on disk: 1335

stage 2, plain: IoU by class
                                     crops  mean IoU  share >= 0.50
label                                                              
ASC-H                                   46     0.620          0.739
ASC-US                                 101     0.602 